What is the run time for the leaders was the simple question I asked myself.  
5 hours later this is as close as I got.


Assumption - the benchmark score is 0.500.    
The only justification is that in past competitions kaggle has often submitted the sample_submission as a simple mean or median. 
Since auc is our metric, guessing it would be median. 
Will do a submission later just for giggles using median of the gold, 
but I thinking kaggle might have cheated and did the full private test.

My run time looks about right, but I don't watch it that close.

Run for the 8/28 revision to the list.

In [ ]:
import pandas as pd
import numpy as np
import os

def calculate_exact_runtimes(
    local_filepath="/home/james/Documents/full_leaderboard.txt",
    kaggle_url="/kaggle/input/notebooks/ryanholbrook/rsna-knee-abnormalities-efficiency-lb/full_leaderboard.csv"
):
    # Determine if the script is running inside a Kaggle environment
    # Kaggle always sets specific environment variables like KAGGLE_KERNEL_RUN_TYPE
    is_kaggle = os.environ.get('KAGGLE_KERNEL_RUN_TYPE') is not None
    
    # Set up the logic to load the file depending on the environment
    if is_kaggle:
        print("Kaggle environment detected.")
        # When running on Kaggle, it is best to attach the efficiency notebook output as a dataset.
        # This is the standard path structure when you add another notebook's output to your current notebook.
        kaggle_input_path = "../input/rsna-knee-abnormalities-efficiency-lb/full_leaderboard.txt"
        
        # Check if the file is attached locally in the Kaggle environment
        if os.path.exists(kaggle_input_path):
            print(f"Reading file from Kaggle input path: {kaggle_input_path}")
            df = pd.read_csv(kaggle_input_path)
        else:
            # Fallback to the direct URL if the dataset is not attached
            # Note: Pandas read_csv can sometimes fail on Kaggle URLs if they render as HTML instead of raw text
            print(f"Reading file directly from Kaggle URL: {kaggle_url}")
            df = pd.read_csv(kaggle_url)
            
        # Define where the output file should be saved in Kaggle (must be in /kaggle/working/)
        output_path = "/kaggle/working/leaderboard_realistic_runtimes_2.csv"
        
    else:
        print("Local Linux environment detected.")
        # Check if the file exists at the specified absolute local path
        if os.path.exists(local_filepath):
            print(f"Reading file from local path: {local_filepath}")
            df = pd.read_csv(local_filepath)
        # Fallback to the current working directory if the absolute path fails
        elif os.path.exists("full_leaderboard.txt"):
            print("Reading file from current working directory.")
            df = pd.read_csv("full_leaderboard.txt")
        else:
            raise FileNotFoundError(f"Could not find the leaderboard file at {local_filepath}")
            
        # Define where the output file should be saved locally
        output_path = "/home/james/Documents/leaderboard_realistic_runtimes_2.csv"

    # Strip any leading or trailing spaces from column names to avoid key errors
    df.columns = df.columns.str.strip()
    
    # Extract the PublicScore column as floating point numbers
    auc = df["PublicScore"].astype(float).values
    
    # Extract the expected ranks as integers
    expected_ranks = df["EfficiencyRank"].astype(int).values
    
    # Get the total number of teams in the dataset
    n = len(auc)
    
    # Dynamically calculate the maximum AUC directly from the current file
    max_auc = auc.max()
    
    # Set the baseline benchmark score for random guessing
    benchmark = 0.500
    
    # Calculate the absolute difference for the denominator
    denom_abs = max_auc - benchmark 
    
    # Calculate the actual formula denominator (negative value)
    denom_formula = benchmark - max_auc 
    
    # Initialize an array to hold the calculated runtimes (starts with all zeros)
    runtimes = np.zeros(n)
    
    # Define a tiny value to break ties strictly when ranks are different
    epsilon = 1e-4
    
    # Loop through all teams starting from the second one (index 1)
    for i in range(1, n):
        # Calculate the mathematical time limit between the current and previous team
        limit = 32400 * (auc[i] - auc[i-1]) / denom_abs
        
        # If the current team has the same rank as the previous team (a tie)
        if expected_ranks[i] == expected_ranks[i-1]:
            # Set their runtime to exactly match the limit boundary to keep the score equal
            runtimes[i] = runtimes[i-1] + limit
        else:
            # If they have a different rank, strictly enforce a higher time penalty
            runtimes[i] = max(0, runtimes[i-1] + limit + epsilon)
            
    # Calculate how much we can shift the runtimes without exceeding the 9-hour limit
    max_shift = 32400 - runtimes.max()
    
    # Shift runtimes up by a maximum of 1000 seconds to avoid 0.0 second runtimes
    shift_amount = min(1000, max_shift - 1)
    runtimes += shift_amount
    
    # Add the used benchmark and max AUC to the dataframe for tracking
    df["Benchmark_Used"] = benchmark
    df["Max_AUC_Used"] = max_auc
    
    # Round the final estimated runtimes to 1 decimal place
    df["Estimated_Runtime_Sec"] = np.round(runtimes, 1)
    
    # Define a helper function to format seconds into HH:MM:SS.s
    def format_time(seconds):
        # Calculate total hours
        h = int(seconds // 3600)
        # Calculate remaining minutes
        m = int((seconds % 3600) // 60)
        # Calculate remaining seconds
        s = seconds % 60
        # Return formatted string
        return f"{h:02d}:{m:02d}:{s:04.1f}"
        
    # Apply the formatting function to the runtime column
    df["Estimated_Time"] = df["Estimated_Runtime_Sec"].apply(format_time)
    
    # Recalculate the exact efficiency score using the official formula
    E_exact = (auc / denom_formula) + (runtimes / 32400.0)
    
    # Round to 9 decimal places to prevent float precision errors from breaking ties
    E_exact_rounded = np.round(E_exact, 9)
    df["Calculated_EfficiencyScore"] = E_exact_rounded
    
    # Rank the recalculated scores, ensuring ties get the same rank (method='min')
    df["Calculated_EfficiencyRank"] = df["Calculated_EfficiencyScore"].rank(ascending=True, method="min").astype(int)
    
    # Store the original ranks for easy comparison
    df["Initial_EfficiencyRank"] = expected_ranks
    
    # Count how many calculated ranks match the original expected ranks
    match_count = np.sum(df["Calculated_EfficiencyRank"].values == expected_ranks)
    print(f"Match Verification: {match_count} / {n} ranks perfectly matched.")
    
    # Define the exact columns we want in our final output file
    output_columns = [
        "Initial_EfficiencyRank",
        "Calculated_EfficiencyRank",
        "TeamName",
        "PublicScore",
        "Calculated_EfficiencyScore",
        "Estimated_Runtime_Sec",
        "Estimated_Time",
        "Benchmark_Used",
        "Max_AUC_Used"
    ]
    
    # Filter the dataframe to only include the specified output columns
    df_output = df[[col for col in output_columns if col in df.columns]]
    
    # Save the dataframe to the designated environment path without the index column
    df_output.to_csv(output_path, index=False)
    print(f"Results saved to {output_path}")
    
    # Return the final dataframe
    return df_output

if __name__ == "__main__":
    # Execute the main function
    results = calculate_exact_runtimes()
    
    # Print a preview of the top 15 teams
    print("\nTop 15 Teams Runtimes:")
    preview_cols = ["Initial_EfficiencyRank", "Calculated_EfficiencyRank", "TeamName", "PublicScore", "Estimated_Time", "Max_AUC_Used"]
    print(results[preview_cols].head(15))